In [1]:
import os
import random
import sys
import time
from argparse import Namespace
from pathlib import Path

import numpy as np
import torch

# Run from the RLinear project root so dataset/checkpoint paths resolve correctly.
RLINEAR_ROOT = (Path.cwd().parent / "models" / "rlinear").resolve()
os.chdir(RLINEAR_ROOT)
sys.path.insert(0, str(RLINEAR_ROOT))


In [3]:
from exp.exp_main import Exp_Main

# User-specified arguments; everything else uses run.py defaults.
args = Namespace(
    # user-specified
    is_training=1,
    data_path="electricity.csv",
    data="custom",
    features="M",
    channel=321,
    itr=1,
    learning_rate=0.005,
    model="RLinear",
    rev=True,
    seq_len=96,
    pred_len=96,
    drop=0,
    # defaults from run.py
    root_path="../../datasets/",
    target="OT",
    freq="h",
    checkpoints="./checkpoints/",
    individual=False,
    seg=20,
    d_model=512,
    layers=2,
    embed="timeF",
    do_predict=False,
    num_workers=10,
    train_epochs=5,
    batch_size=16,
    patience=10,
    lradj="type1",
    use_amp=False,
    use_gpu=True,
    gpu=0,
    use_multi_gpu=False,
    devices="0,1,2,3",
)

fix_seed = 1024
random.seed(fix_seed)
torch.manual_seed(fix_seed)
np.random.seed(fix_seed)

args.use_gpu = True if torch.cuda.is_available() and args.use_gpu else False

if args.use_gpu and args.use_multi_gpu:
    args.devices = args.devices.replace(" ", "")
    device_ids = args.devices.split(",")
    args.device_ids = [int(id_) for id_ in device_ids]
    args.gpu = args.device_ids[0]

print("Args in experiment:")
print(args)

Exp = Exp_Main

if args.is_training:
    for ii in range(args.itr):
        setting = "{}_{}_ft{}_sl{}_pl{}_dm{}_eb{}_{}".format(
            args.model,
            args.data_path[:-4],
            args.features,
            args.seq_len,
            args.pred_len,
            args.d_model,
            args.embed,
            ii,
        )

        exp = Exp(args)
        print(">>>>>>>start training : {}>>>>>>>>>>>>>>>>>>>>>>>>>>".format(setting))
        exp.train(setting)

        time_now = time.time()
        print(">>>>>>>testing : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<".format(setting))
        exp.test(setting)
        print("Inference time: ", time.time() - time_now)

        if args.do_predict:
            print(">>>>>>>predicting : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<".format(setting))
            exp.predict(setting, True)

        torch.cuda.empty_cache()
else:
    ii = 0
    setting = "{}_{}_ft{}_sl{}_pl{}_dm{}_eb{}_{}".format(
        args.model,
        args.data_path[:-4],
        args.features,
        args.seq_len,
        args.pred_len,
        args.d_model,
        args.embed,
        ii,
    )

    exp = Exp(args)
    print(">>>>>>>testing : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<".format(setting))
    exp.test(setting, test=1)
    torch.cuda.empty_cache()


Args in experiment:
Namespace(is_training=1, data_path='electricity.csv', data='custom', features='M', channel=321, itr=1, learning_rate=0.005, model='RLinear', rev=True, seq_len=96, pred_len=96, drop=0, root_path='../../datasets/', target='OT', freq='h', checkpoints='./checkpoints/', individual=False, seg=20, d_model=512, layers=2, embed='timeF', do_predict=False, num_workers=10, train_epochs=5, batch_size=16, patience=10, lradj='type1', use_amp=False, use_gpu=True, gpu=0, use_multi_gpu=False, devices='0,1,2,3')
Use GPU: cuda:0
Number of parameters: 0.01M
>>>>>>>start training : RLinear_electricity_ftM_sl96_pl96_dm512_ebtimeF_0>>>>>>>>>>>>>>>>>>>>>>>>>>
train 18221
val 2537
test 5165
	iters: 100, epoch: 1 | loss: 0.1908224
	speed: 0.0095s/iter; left time: 53.1540s
	iters: 200, epoch: 1 | loss: 0.1882070
	speed: 0.0072s/iter; left time: 39.6481s
	iters: 300, epoch: 1 | loss: 0.2397005
	speed: 0.0072s/iter; left time: 38.5503s
	iters: 400, epoch: 1 | loss: 0.1985814
	speed: 0.0071s/iter